# Inference + BoT-SORT Tracking (Thermal or RGB)

This notebook runs detector inference and BoT-SORT tracking for either detector variant, then previews output frames.


In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('IN_COLAB =', IN_COLAB)


In [ ]:
PROJECT_ROOT = Path('/content/VisionSentry') if IN_COLAB and Path('/content/VisionSentry').exists() else Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError(f'Cannot find project at {PROJECT_ROOT}. Set PROJECT_ROOT to your repo path.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
MODALITY = 'rgb'  # 'ir' or 'rgb'
RUN_NAME = 'yolo12n_rgb_uav' if MODALITY == 'rgb' else 'yolo12n_thermal_uav'
weights_path = PROJECT_ROOT / 'runs' / 'detect' / RUN_NAME / 'weights' / 'best.pt'
if not weights_path.exists() and MODALITY == 'ir':
    weights_path = PROJECT_ROOT / 'weights' / 'best.pt'
source_path = PROJECT_ROOT / 'data' / 'sample.mp4'
tracker_cfg = PROJECT_ROOT / 'configs' / 'tracker_botsort.yaml'

print('weights:', weights_path)
print('source:', source_path)
print('tracker:', tracker_cfg)


In [ ]:
detect_cmd = [
    sys.executable, '-m', 'src.detection.infer',
    '--weights', str(weights_path),
    '--source', str(source_path),
    '--imgsz', '960',
    '--conf', '0.25',
    '--iou', '0.5',
    '--device', '0',
    '--save_video', 'true',
    '--save_frames', 'false',
    '--save_txt', 'true',
    '--project', 'runs/predict',
    '--name', f'{MODALITY}_detect_notebook',
]
print('Running:', ' '.join(detect_cmd))
subprocess.check_call(detect_cmd, cwd=PROJECT_ROOT)


In [ ]:
track_cmd = [
    sys.executable, '-m', 'src.tracking.run_tracker',
    '--weights', str(weights_path),
    '--source', str(source_path),
    '--tracker', str(tracker_cfg),
    '--imgsz', '960',
    '--conf', '0.25',
    '--iou', '0.5',
    '--device', '0',
    '--save_video', 'true',
    '--save_frames', 'true',
    '--with_reid', 'false',
    '--project', 'runs/track',
    '--name', f'{MODALITY}_track_notebook',
]
print('Running:', ' '.join(track_cmd))
subprocess.check_call(track_cmd, cwd=PROJECT_ROOT)


In [ ]:
import matplotlib.pyplot as plt
from src.utils.visualization import load_preview_images

track_runs = sorted((PROJECT_ROOT / 'runs' / 'track').glob(f'{MODALITY}_track_notebook*'))
if not track_runs:
    raise FileNotFoundError('No tracking runs found under runs/track')
latest_track_run = track_runs[-1]
frame_dir = latest_track_run / 'frames'
images = load_preview_images(frame_dir, max_images=4)

print('Latest run:', latest_track_run)
print('MOT file:', latest_track_run / 'tracks_mot.txt')

if not images:
    print('No preview frames found.')
else:
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    if len(images) == 1:
        axes = [axes]
    for i, image in enumerate(images):
        axes[i].imshow(image)
        axes[i].set_title(f'Frame {i + 1}')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
